# RTIH RAG Starter — build a memory for your AI

The five-step pipeline: **LOAD → CHUNK → EMBED → STORE → RETRIEVE → answer**.

Run each cell top to bottom. You edit **three things**: your data, your question, and which model answers.

Embeddings run locally (no key). The final answer step goes through **OpenRouter** — one key, and any model (Claude, GPT, open-source) is a one-line swap.

### Step 0 — install the libraries (run once, ~1 min)

> **You'll see red "dependency conflict" warnings — that's normal, ignore them.** They come from Colab's pre-installed Google packages, not from this notebook. The install still works.
>
> *Rare:* if a later cell errors mentioning `chromadb` or `opentelemetry`, do **Runtime → Restart session**, then re-run every cell **except** this install one.

In [ ]:
!pip -q install langchain langchain-community langchain-text-splitters langchain-chroma langchain-openai langchain-anthropic fastembed

### Step 0b — paste your API key
Pick your provider — **OpenRouter** (default, one key for every model), **Anthropic**, or **OpenAI**. Paste the key for whichever you chose. It will be hidden as you type.

In [ ]:
import os, getpass

# Default: OpenRouter. Get a key at https://openrouter.ai/keys
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key: ')

# Using Anthropic or OpenAI instead? Comment the line above and uncomment one below:
# os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Paste your Anthropic API key: ')
# os.environ['OPENAI_API_KEY'] = getpass.getpass('Paste your OpenAI API key: ')

### >>> DROP YOUR DATA HERE

**Where does my file go?** Colab is a cloud machine — it **cannot see your laptop**. Upload your files into it first:
1. Click the **folder icon** on the left sidebar.
2. Make a folder called **`data`**.
3. **Upload** your `.txt` / `.md` files into that `data` folder.

Then just **run the cell below** — it's safe either way: if it finds your uploaded files it uses them and writes nothing; if `data/` is empty it drops in a built-in sample so the notebook still works.

> Uploaded files are **temporary** — they vanish when the runtime disconnects. Re-upload if you come back later.

In [ ]:
import os, glob
os.makedirs('data', exist_ok=True)

# Smart: only write the built-in sample if you HAVEN'T uploaded your own files.
# So this cell is always safe to run — it never overwrites your data.
existing = [f for f in glob.glob('data/**/*', recursive=True) if os.path.isfile(f)]

if existing:
    print(f"Found {len(existing)} file(s) you uploaded in data/ — using those:")
    for f in existing:
        print("  -", f)
    print("(Sample NOT written. Skipping ahead is fine.)")
else:
    with open('data/sample_emails.txt', 'w') as f:
        f.write('''=== EMAIL 1 ===
From: site.engineer@meadowsdev.com
Date: 2026-05-08
Subject: What the Tower B electrical inspection will cover
An electrical inspection verifies that the building's wiring is safe and code-compliant. It covers: earthing and grounding of the whole installation; RCD/RCCB protection on every circuit; the integrity of the main and sub distribution boards; cable insulation-resistance testing; correct circuit labelling and load balancing; and emergency lighting and exit signage. If any item fails, the inspector withholds the completion certificate until it is fixed and re-checked.

=== EMAIL 2 ===
From: site.engineer@meadowsdev.com
Date: 2026-05-12
Subject: Tower B - electrical inspection result
The electrical inspection for Tower B was completed on May 11. The inspector flagged two issues: the basement distribution board needs additional earthing, and three units on floor 7 are missing RCD protection. We cannot get the completion certificate until both are fixed. Estimated rework time is 4 days.

=== EMAIL 3 ===
From: pm@meadowsdev.com
Date: 2026-05-14
Subject: Handover timeline update - Phase 1
Following the electrical re-inspection, the Tower B handover date moves from June 1 to June 8. Tower A is unaffected and remains on track for June 1. We will confirm the final Tower B date once the completion certificate is issued.

=== EMAIL 4 ===
From: procurement@meadowsdev.com
Date: 2026-05-15
Subject: Lift installation - vendor delay
Otis has informed us the lift cars for Tower A will arrive two weeks late due to a customs hold at Chennai port. Installation takes 6 working days after delivery, putting lift commissioning at approximately June 9. This may affect the Tower A handover.

=== EMAIL 5 ===
From: pm@meadowsdev.com
Date: 2026-05-16
Subject: Budget variance - Phase 1 to date
Phase 1 is currently 6.5% over the approved budget. The two largest drivers are the electrical rework on Tower B (approx 8 lakh) and additional waterproofing on the podium (approx 14 lakh) that was not in the original scope. We need a revised budget sign-off before the next contractor payment.

=== EMAIL 6 ===
From: client@parkviewholdings.com
Date: 2026-05-18
Subject: Re: Handover timeline update - Phase 1
Thanks for the update. June 8 for Tower B is acceptable. Please ensure the lift situation does not push Tower A past June 1 - that date is committed to our first set of buyers. Send me a confirmed lift commissioning date by end of this week.

=== EMAIL 7 ===
From: fire.consultant@meadowsdev.com
Date: 2026-05-19
Subject: Fire NOC status - both towers
The fire NOC for both Tower A and Tower B was approved by the fire department on May 17. Sprinklers and wet risers have been pressure-tested and passed. The emergency lighting on Tower B is tied to the electrical re-inspection, so Tower B's fire clearance is only final once the electrical completion certificate is issued.

=== EMAIL 8 ===
From: facilities@meadowsdev.com
Date: 2026-05-21
Subject: Tower A snagging list
The Tower A snagging walk logged 18 minor snags: paint touch-ups in common corridors, two misaligned apartment doors, and a leaking kitchen tap in four units. All are cosmetic and do not affect the June 1 handover. The contractor will close them out by May 28.

=== EMAIL 9 ===
From: electrical.inspector@apinspections.gov.in
Date: 2026-05-22
Subject: Tower B re-inspection result - PASSED
The Tower B re-inspection on May 22 passed. The basement earthing and the floor 7 RCD protection have both been verified and now meet code. No further electrical issues were found. The completion certificate for Tower B will be issued on May 25.

=== EMAIL 10 ===
From: site.engineer@meadowsdev.com
Date: 2026-05-20
Subject: Tower B - rework complete
Both electrical issues on Tower B are resolved. The basement earthing was completed on May 19 and the floor 7 RCDs were installed today. Re-inspection is booked for May 22, keeping us on track for the June 8 handover.

=== EMAIL 11 ===
From: pm@meadowsdev.com
Date: 2026-05-26
Subject: Tower B completion certificate issued
The Tower B electrical completion certificate was issued on May 25, and the fire clearance is now final as a result. Tower B is firmly confirmed for the June 8 handover. Tower A remains on June 1.

=== EMAIL 12 ===
From: procurement@meadowsdev.com
Date: 2026-05-28
Subject: Lift commissioning - confirmed
The Otis cars cleared customs on May 27 and installation has started. Commissioning is now confirmed for June 7, one day ahead of need. This keeps the Tower A handover safe on June 1.

=== EMAIL 13 ===
From: finance@meadowsdev.com
Date: 2026-05-29
Subject: Revised budget signed off
The board approved the revised Phase 1 budget, accepting the 6.5% variance. The next contractor payment is released. They asked us to treat the podium waterproofing (approx 14 lakh) as out-of-scope and open a cost-recovery review against the original waterproofing contractor.

=== EMAIL 14 ===
From: client@parkviewholdings.com
Date: 2026-05-30
Subject: Clubhouse and amenities handover
Before our first residents move in around mid-June, we need the shared amenities ready. When will the clubhouse, the gym, and the landscaped podium be handed over? Please give me firm dates.

=== EMAIL 15 ===
From: pm@meadowsdev.com
Date: 2026-06-01
Subject: Re: Clubhouse and amenities handover
Amenities timeline: the clubhouse and gym will be ready on June 12, and the landscaped podium on June 14. Both are ahead of the mid-June move-ins. The swimming pool is the only item still pending a water-treatment clearance, which I will confirm by June 5.
''')
    print("No files found in data/ — wrote built-in sample (15 emails) so the notebook works out of the box.")

### Steps 1–5 — the whole pipeline

> **First run downloads the embedding model (~67MB) — give it up to a minute.** You'll see a progress bar and some yellow warnings (`langchain-community sunset`, `HF_TOKEN does not exist`) — ignore them, they don't stop anything.
>
> **Downloads slow or stalling?** Add your HuggingFace token to Colab: left sidebar **🔑 Secrets → New secret**, name it `HF_TOKEN`, paste your `hf_...` token, turn **notebook access on**, then re-run this cell. (Same token as the no-code / Flowise track.)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI  # OpenRouter speaks the OpenAI dialect

# 1. LOAD — this is the ONLY step that changes per data type. Pick ONE.
docs = DirectoryLoader('data/', glob='**/*.txt', loader_cls=TextLoader).load()   # .txt (default)
# docs = DirectoryLoader('data/', glob='**/*.md', loader_cls=TextLoader).load()  # Markdown

# PDFs (run: !pip install pypdf):
# from langchain_community.document_loaders import PyPDFDirectoryLoader
# docs = PyPDFDirectoryLoader('data/').load()

# Word .docx (run: !pip install docx2txt):
# from langchain_community.document_loaders import Docx2txtLoader
# docs = Docx2txtLoader('data/file.docx').load()

# CSV / spreadsheet:
# from langchain_community.document_loaders import CSVLoader
# docs = CSVLoader('data/data.csv').load()

# Logs + profiles / any table (run: !pip install pandas):
# import pandas as pd
# from langchain_community.document_loaders import DataFrameLoader
# df = pd.read_csv('data/members.csv')
# df['text'] = df.apply(lambda r: f"Member {r['name']}: {r['notes']}", axis=1)
# docs = DataFrameLoader(df, page_content_column='text').load()

# Universal fallback — any weird format, build documents yourself:
# from langchain_core.documents import Document
# docs = [Document(page_content=text, metadata={'source': name}) for name, text in my_records]

# 2. CHUNK
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)

# 3. EMBED + 4. STORE
db = Chroma.from_documents(chunks, FastEmbedEmbeddings())

# 5. RETRIEVE
retriever = db.as_retriever(search_kwargs={'k': 4})
print(f'Loaded {len(docs)} docs, split into {len(chunks)} chunks. Ready.')

### >>> ASK YOUR QUESTION HERE
Change the question to a real query your product needs to answer, then run.

In [ ]:
QUESTION = 'What is the new handover date for Tower B and why did it change?'

context = '\n\n'.join(d.page_content for d in retriever.invoke(QUESTION))

# >>> PICK YOUR PROVIDER — uncomment ONE option. You only need one key.
# Option A (default): OpenRouter — one key for every model. Change MODEL to swap brains.
#   Browse models at https://openrouter.ai/models
MODEL = 'anthropic/claude-haiku-4.5'   # or 'openai/gpt-4o-mini', 'meta-llama/llama-3.3-70b-instruct'
llm = ChatOpenAI(model=MODEL, base_url='https://openrouter.ai/api/v1',
                 api_key=os.environ['OPENROUTER_API_KEY'],
                 max_tokens=512)  # cap output: OpenRouter reserves credit for max_tokens;
                                  # without this a small balance gets a 402 error.

# Option B: Anthropic directly (set ANTHROPIC_API_KEY above)
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model='claude-haiku-4-5', max_tokens=512)

# Option C: OpenAI directly (set OPENAI_API_KEY above)
# llm = ChatOpenAI(model='gpt-4o-mini', max_tokens=512)

prompt = ('Answer using ONLY the context below. '
          "If the answer is not in the context, say 'I don't know based on the provided data.'\n\n"
          f'Context:\n{context}\n\nQuestion: {QUESTION}')

print('ANSWER:', llm.invoke(prompt).content)

### Ask your own questions (a mini chat)

Run the cell below and type questions one after another — **test your data freely**, no need to edit code. Type `quit` to stop.

Each question is answered fresh from your data (it doesn't remember previous questions — that's fine for testing). Try one your data *can't* answer and watch it say "I don't know."

In [ ]:
# Ask as many questions as you want — type 'quit' (or leave blank) to stop.
# Reuses the retriever + llm from the cell above. Each question is answered fresh.
def answer(question):
    context = '\n\n'.join(d.page_content for d in retriever.invoke(question))
    prompt = ('Answer using ONLY the context below. '
              "If the answer is not in the context, say 'I don't know based on the provided data.'\n\n"
              f'Context:\n{context}\n\nQuestion: {question}')
    return llm.invoke(prompt).content

while True:
    q = input("\nAsk a question (or 'quit'): ").strip()
    if q.lower() in ('quit', 'exit', 'q', ''):
        print('Done.')
        break
    print('>', answer(q))

### Now try to break it
- Ask something the data **doesn't** contain — does it say *I don't know*, or make it up?
- Set `chunk_size=2000` and re-run — does retrieval get worse?

If you get a grounded answer from your own data, you've hit the bar for Friday office hours.